#### Loss Function(손실함수)
: 모델의 예측값이 실제 정답과 얼마나 다른지 수치로 계산하는 함수<br>
모델의 목표는 손실값을 줄이는 것

- 기본용어
    - Prediction : 모델이 예측한 값
    - Target : 정답
    - Loss : 예측값과 정답의 차이를 계산한 값

- 손실함수의 역할
    - 예측값과 정답의 차이를 수치로 표현
    - 모델이 얼마나 잘 예측했는지 평가할 수 있음
    - 자동 미분을 통해 weight와 bias를 업데이트 할 때 사용할 기울기 계산 가능

#### 1. MSELoss(Mean Squared Error)
: 평균의 제곱 오차를 계산하는 손실함수<br>
예측값과 정답이 차이를 제곱한 뒤 평균을 계산함

- 특징
    - 오차를 제곱하므로 큰 오차에 더 큰 패널티를 부여함
    - 손실값이 0 이상임
    - 예측값이 정답과 같으면 손실값은 0
    - 주로 회귀문제에서 사용

- 사용방법
loss_fn = nn.MSELoss()
loss = loss_fn(prediction, target)

기본설정 reduction='mean'은 모든 원소의 제곱 오차의 평균

In [2]:
import torch
from torch import nn

In [5]:
prediction = torch.tensor([2., 4, 6])
target = torch.tensor([1.,5,4])

loss_fn = nn.MSELoss()
loss = loss_fn(prediction, target)

print('예측값 :', prediction)
print('정답 :', target)
print('MSE Loss :', loss)

loss2 = ((target - prediction)**2).mean()
print('직접 계산한 Loss', loss2)

예측값 : tensor([2., 4., 6.])
정답 : tensor([1., 5., 4.])
MSE Loss : tensor(2.)
직접 계산한 Loss tensor(2.)


#### - reduction 옵션
| 옵션 | 의미 |
| --- | --- |
| mean | 모든 원소의 손실 평균(기본값) |
| sum | 모든 원소의 손실 합 |
| none | 원소별 손실을 그대로 반환 |


** none은 데이터별 손실을 따로 확인하거나 가중치를 적용할 때 유용함

In [12]:
prediction = torch.tensor([2.,4,6])
target = torch.tensor([1,5., 4])

loss_mean = nn.MSELoss(reduction='mean')(prediction, target)
loss_sum = nn.MSELoss(reduction='sum')(prediction, target)
loss_none = nn.MSELoss(reduction='none')(prediction, target)

print('mean :',loss_mean)
print('sum :', loss_sum)
print('none :', loss_none)

mean : tensor(2.)
sum : tensor(6.)
none : tensor([1., 1., 4.])


#### L1Loss(Mean Absolute Error)
: 예측값과 정답의 차를 절댓값을 적용한 뒤 평균 계산

- 특징
    - 오차에 절대값 사용
    - 손실값은 0이상
    - 예측값과 정답이 같으면 손실 값은 0
    - MSE보다 큰 오차에 대한 패널티가 상대적으로 작음
    - 주로 회귀문제에서 사용
- 사용법<br>
loss_fn = nn.L1Loss()<br>
loss = loss_fn(prediction, target)

In [8]:
prediction = torch.tensor([2., 4, 6])
target = torch.tensor([1.,4,5])

loss_fn = nn.L1Loss()
loss = loss_fn(prediction, target)

print('예측값 :', prediction)
print('정답 :', target)
print('L1 Loss :', loss)

loss2 = torch.abs(target-prediction).mean()
print('직접 계산한 MAE :', loss2)

예측값 : tensor([2., 4., 6.])
정답 : tensor([1., 4., 5.])
L1 Loss : tensor(0.6667)
직접 계산한 MAE : tensor(0.6667)


- 큰 오차를 더 강하게 반영하고 싶다면 MSE를, 큰 오차나 이상치의 영향을 상대적으로 적게 받고 싶으면 L1을 고려

In [9]:
prediction = torch.tensor([10., 10, 10])
target = torch.tensor([9.,8,0])

mse_loss = nn.MSELoss()(prediction, target)
l1_loss = nn.L1Loss()(prediction, target)

print('MSE Loss :', mse_loss)
print('L1 Loss :', l1_loss)

MSE Loss : tensor(35.)
L1 Loss : tensor(4.3333)


#### 3. BCELoss(Binary Cross Entropy)
: 이진분류에서 사용하는 손실 함수

**중요**<br>
모델이 Sigmoid를 통과한 확률값을 입력으로 받아야 함.<br><br>

nn.BCELoss()(Probability, target)<br>
&emsp;Probability : 0~1사이의 확률값<br>
&emsp;Target : 정답. 일반적으로 0또는 1이며 실수형 텐서로 전달함<br><br>

- 실제 정답이 1인데 확률이 1에 가까우면 손실이 작아지고, 0에 가까워지면 손실이 커짐
- 극단적인 값에서 NaN 발생 위험 있음

In [10]:
probability = torch.tensor([0.9,0.2,0.7,0.1])
target = torch.tensor([1.,0,1,0])

loss = nn.BCELoss()(probability, target)

print('예측 확률 :', probability)
print('정답 :', target)
print('BCE Loss :', loss)

예측 확률 : tensor([0.9000, 0.2000, 0.7000, 0.1000])
정답 : tensor([1., 0., 1., 0.])
BCE Loss : tensor(0.1976)


#### 4. BCEWithLogitsLoss
: Sigmoid와 BCELoss 계산을 하나로 결합한 손실함수

- 중요한 차이
    - BCELoss() : Sigmoid를 적용한 확률값을 입력으로 받음
    - BCEWithLogitsLoss() : Sigmoid를 적용하지 않은 Logits을 입력으로 받음
- 사용법<br>
nn.BCEWithLogitsLoss()(logits, target)

- 왜 사용하나<br>
Sigmoid와 BCELoss계산을 결합하여 수치적으로 더 안정적인 방식으로 손실을 계산할 수 있음<br>
내부적으로 안정적인 연산 수행(극단적인 값에서 NaN 발생 위험이 적음)

In [11]:
logits = torch.tensor([2, -1.5, 1, -2])
target = torch.tensor([1.,0,1,0])

loss = nn.BCEWithLogitsLoss()(logits, target)

print('Logits :', logits)
print('BCEWithLogitsLoss :', loss)

Logits : tensor([ 2.0000, -1.5000,  1.0000, -2.0000])
BCEWithLogitsLoss : tensor(0.1921)


In [16]:
logits = torch.tensor([1.2, -0.7, 2, -1.5])
target = torch.tensor([1.,0,1,0])

# BCELoss
prob = torch.sigmoid(logits)
loss_b = nn.BCELoss()(prob, target)

loss_l = nn.BCEWithLogitsLoss()(logits, target)

print('BCELoss :', loss_b)
print('BCEWithLogitsLoss :', loss_l)
# 두 손실값이 같은지
print('두 손실값이 같은가?', torch.allclose(loss_b,loss_l))

BCELoss : tensor(0.2487)
BCEWithLogitsLoss : tensor(0.2487)
두 손실값이 같은가? True


In [18]:
logits_a = torch.tensor([1000.], requires_grad=True)
target = torch.tensor([0.])

probability = torch.sigmoid(logits_a)
loss_a = nn.BCELoss()(probability, target)

loss_a.backward()

logits_b = torch.tensor([1000.], requires_grad=True)
loss_b = nn.BCEWithLogitsLoss()(logits_b, target)

print(loss_a)
print(loss_b)

tensor(100., grad_fn=<BinaryCrossEntropyBackward0>)
tensor(1000., grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)


In [ ]:
# Loss값이 차이나고, 기울기가 소실되는 예
target = torch.tensor([0.])

logits_a = torch.tensor([1000.], requires_grad=True)

probability = torch.sigmoid(logits_a)
loss_a = nn.BCELoss()(probability, target)

loss_a.backward()

logits_b = torch.tensor([1000.], requires_grad=True)
loss_b = nn.BCEWithLogitsLoss()(logits_b, target)

loss_b.backward()

print('Sigmoid 확률 :',probability.item())
print('BCE Loss :',loss_a.item())
print('BCEWithLogitsLoss :',loss_b.item())
print('BCE Loss의 기울기 :', logits_a.grad.item()) #기울기 소실로 0.0 출력
print('BCEWithLogitsLoss의 기울기 :', logits_b.grad.item())

Sigmoid 확률 : 1.0
BCE Loss : 100.0
BCEWithLogitsLoss : 1000.0
BCE Loss의 기울기 : 0.0
BCEWithLogitsLoss의 기울기 : 1.0


#### 5. CrossEntropyLoss
: 여러 클래스 중 하나를 선택하는 다중 분류에서 주로 사용하는 손실 함수

- 사용방법
nn.CrossEntropyLoss()(logits, target)

- 입력형식
    - 모델 출력(logits) : (batch_size, num_classes)
    - 정답(target) : (batch_size)
    - 정답의 자료형 : torch.long
- 내부적으로 LogSoftmax와 NLLLoss에 해당하는 계산을 수행
    - LogSoftmax : softmax 결과에 log를 적용한 함수
    - NLLLoss : 그 로그확률을 이용하여 손실을 계산하는 함수

In [7]:
# 데이터 3개, 클래스 4개에 대한 모델 출력이 되었음(logits)
logits = torch.tensor([[2.,1, 0.1, 0.5], [0.2, 2.5, 1, 0.4], [0.1, 0.4, 0.8, 2]])

# 각 데이터의 정답 클래스 인덱스
target = torch.tensor([0,1,3], dtype=torch.long)
loss = nn.CrossEntropyLoss()(logits, target)
print('CrossEntropy Loss :',loss)

CrossEntropy Loss : tensor(0.4751)


#### 문제 유형에 따른 손실함수 선택
| 문제 유형 | 대표 손실함수 | 모델 선택 |
| --- | --- | --- |
| 회귀 | MSELoss, L1Loss | 일반적인 실수값 |
| 이진분류 | BCELoss | Sigmoid 적용 확률 |
| 이진분류 | BCEWithLogitsLoss | Sigmoid 적용 전 logits |
| 다중분류 | CrossEntropyLoss | Sigmoid 적용 전 logits |

In [11]:
# 각 손실함수의 입력 형태에 따른 확인

print('회귀')
regression_prediction = torch.tensor([2.5, 3])
regression_target = torch.tensor([3,2.])

print('MSELoss :', nn.MSELoss()(regression_prediction, regression_target).item())
print('L1Loss :', nn.L1Loss()(regression_prediction, regression_target).item())

print('이진분류')
binary_logits = torch.tensor([1.,-1])
binary_target = torch.tensor([1,0.])

print('BCEWithLogitsLoss :', nn.BCEWithLogitsLoss()(binary_logits, binary_target).item())

print('다중분류')
multi_logits = torch.tensor([[1,2, 0.5],[2, 0.5, 1]])
multi_target = torch.tensor([1, 0], dtype=torch.long)

print('CrossEntropyLoss :', nn.CrossEntropyLoss()(multi_logits, multi_target).item())

회귀
MSELoss : 0.625
L1Loss : 0.75
이진분류
BCEWithLogitsLoss : 0.3132617175579071
다중분류
CrossEntropyLoss : 0.4643687903881073
